In [2]:
!pip install -q mediapipe==0.10.35

In [3]:
import os, subprocess, json
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists("/content/raw"):
    subprocess.run(["unzip","-q","-o","/content/drive/MyDrive/raw_videos.zip","-d","/content/"], check=True)
metadata = json.load(open("/content/drive/MyDrive/metadata.json"))
split = json.load(open("/content/drive/MyDrive/split_v1.json"))
def vid_id(name): return name.replace(".mp4","")
label_of = {vid_id(n): (0 if i["label"]=="REAL" else 1) for n,i in metadata.items()}
print("Videos found:", len([f for f in os.listdir("/content/raw") if f.endswith(".mp4")]))
print("Split counts:", {k: len(v) for k,v in split.items()})

Mounted at /content/drive
Videos found: 401
Split counts: {'train': 292, 'val': 54, 'test': 54}


In [4]:
import mediapipe as mp, urllib.request, cv2, numpy as np, librosa
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

lm_path = "/content/face_landmarker.task"
if not os.path.exists(lm_path):
    urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task", lm_path)
landmarker = mp_vision.FaceLandmarker.create_from_options(mp_vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=lm_path),
    running_mode=mp_vision.RunningMode.IMAGE, num_faces=1))

det_path = "/content/face_detector.tflite"
if not os.path.exists(det_path):
    try:
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_full_range/float16/1/blaze_face_full_range.tflite", det_path)
        print("Downloaded FULL-range detector")
    except Exception as e:
        print("Using short-range fallback:", e)
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite", det_path)
face_detector = mp_vision.FaceDetector.create_from_options(mp_vision.FaceDetectorOptions(
    base_options=mp_python.BaseOptions(model_asset_path=det_path), min_detection_confidence=0.3))

def detect_and_crop(rgb, margin=0.35):
    h, w, _ = rgb.shape
    res = face_detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(rgb)))
    if not res.detections: return None
    b = res.detections[0].bounding_box
    x1 = max(b.origin_x - int(b.width*margin), 0); y1 = max(b.origin_y - int(b.height*margin), 0)
    x2 = min(b.origin_x + b.width + int(b.width*margin), w); y2 = min(b.origin_y + b.height + int(b.height*margin), h)
    crop = rgb[y1:y2, x1:x2]
    return np.ascontiguousarray(crop) if crop.size > 0 else None
print("Ready")

Downloaded FULL-range detector
Ready


In [5]:
STRIDE = 2   # use every 2nd frame (~15 per second); enough for speech mouth movement

def extract_mouth_series(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    times, mar, i = [], [], 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if i % STRIDE == 0:
            val = np.nan
            crop = detect_and_crop(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if crop is not None:
                res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=crop))
                if res.face_landmarks:
                    lm = res.face_landmarks[0]; h, w, _ = crop.shape
                    P = lambda k: np.array([lm[k].x*w, lm[k].y*h])
                    width = np.linalg.norm(P(61) - P(291))          # mouth corners
                    if width > 1e-3: val = np.linalg.norm(P(13) - P(14)) / width   # inner lips / width
            times.append(i / fps); mar.append(val)
        i += 1
    cap.release()
    return np.array(times), np.array(mar), fps / STRIDE

def audio_envelope(video_path, times):
    raw = subprocess.run(["ffmpeg","-v","error","-i",video_path,"-vn","-ac","1","-ar","16000","-f","f32le","pipe:1"],
                         capture_output=True).stdout
    y = np.frombuffer(raw, dtype=np.float32)
    if len(y) < 1600: return None, 0.0
    hop = 160
    rms = librosa.feature.rms(y=y, frame_length=640, hop_length=hop)[0]
    t_a = np.arange(len(rms)) * hop / 16000
    p95 = np.percentile(rms, 95)
    speech_frac = float(np.mean(rms > 0.1*p95)) if p95 > 1e-4 else 0.0
    return np.interp(times, t_a, np.log(rms + 1e-5)), speech_frac

def sync_features(mar, env, fps_eff, max_lag_s=0.3):
    ok = ~np.isnan(mar)
    if ok.sum() < 20 or len(mar) != len(env): return None
    x = np.interp(np.arange(len(mar)), np.where(ok)[0], mar[ok])
    a = (x - x.mean())/(x.std()+1e-8); b = (env - env.mean())/(env.std()+1e-8)
    L = int(round(max_lag_s*fps_eff)); n = len(a); cc = []
    for lag in range(-L, L+1):
        u, v = (a[lag:], b[:n-lag]) if lag >= 0 else (a[:n+lag], b[-lag:])
        cc.append(float(np.mean(u*v)))
    cc = np.array(cc); lags = np.arange(-L, L+1)/fps_eff
    return dict(c0=cc[L], cmax=cc.max(), cabs=np.abs(cc).max(), lag_best=lags[cc.argmax()],
                cmean=cc.mean(), valid_frac=ok.mean())

def process_video(vid):
    t, mar, fps_eff = extract_mouth_series(f"/content/raw/{vid}.mp4")
    env, speech_frac = audio_envelope(f"/content/raw/{vid}.mp4", t)
    if env is None: return dict(applicable=False, reason="no audio")
    f = sync_features(mar, env, fps_eff)
    if f is None: return dict(applicable=False, reason="no face")
    f.update(applicable=speech_frac > 0.2, speech_frac=speech_frac,
             reason="" if speech_frac > 0.2 else "little speech")
    return f

import time
ids = [v for v in label_of if label_of[v]==0][:2] + [v for v in label_of if label_of[v]==1][:2]
for v in ids:
    t0 = time.time(); r = process_video(v)
    print("REAL" if label_of[v]==0 else "FAKE", v, {k:(round(x,2) if isinstance(x,float) else x) for k,x in r.items()}, f"{time.time()-t0:.0f}s")

REAL abarnvbtwb {'c0': np.float64(0.42), 'cmax': np.float64(0.43), 'cabs': np.float64(0.43), 'lag_best': np.float64(0.07), 'cmean': np.float64(0.4), 'valid_frac': np.float64(1.0), 'applicable': True, 'speech_frac': 1.0, 'reason': ''} 26s
REAL aelfnikyqj {'c0': np.float64(-0.04), 'cmax': np.float64(0.21), 'cabs': np.float64(0.21), 'lag_best': np.float64(-0.13), 'cmean': np.float64(0.0), 'valid_frac': np.float64(1.0), 'applicable': True, 'speech_frac': 1.0, 'reason': ''} 8s
FAKE aagfhgtpmv {'c0': np.float64(-0.07), 'cmax': np.float64(0.33), 'cabs': np.float64(0.33), 'lag_best': np.float64(-0.2), 'cmean': np.float64(0.07), 'valid_frac': np.float64(1.0), 'applicable': True, 'speech_frac': 1.0, 'reason': ''} 9s
FAKE aapnvogymq {'c0': np.float64(-0.02), 'cmax': np.float64(0.02), 'cabs': np.float64(0.13), 'lag_best': np.float64(-0.13), 'cmean': np.float64(-0.04), 'valid_frac': np.float64(1.0), 'applicable': True, 'speech_frac': 0.79, 'reason': ''} 7s


In [7]:
import pickle
from tqdm import tqdm
save_path = "/content/drive/MyDrive/sync_features_v1.pkl"
feats = pickle.load(open(save_path,"rb")) if os.path.exists(save_path) else {}
todo = [v for v in label_of if v not in feats and os.path.exists(f"/content/raw/{v}.mp4")]
print("already done:", len(feats), "| to do:", len(todo))
for n, v in enumerate(tqdm(todo)):
    try: feats[v] = process_video(v)
    except Exception as e: feats[v] = dict(applicable=False, reason=f"error: {e}")
    if n % 20 == 0: pickle.dump(feats, open(save_path,"wb"))
pickle.dump(feats, open(save_path,"wb"))
print("done:", len(feats), "| applicable:", sum(f["applicable"] for f in feats.values()))
from collections import Counter
print(Counter(f["reason"] for f in feats.values() if not f["applicable"]))

already done: 0 | to do: 400


100%|██████████| 400/400 [46:54<00:00,  7.04s/it]

done: 400 | applicable: 397
Counter({'little speech': 3})


In [10]:
import pickle, os
save_path = "/content/drive/MyDrive/sync_features_v1.pkl"
print("file exists:", os.path.exists(save_path))
feats = pickle.load(open(save_path, "rb"))
print("videos:", len(feats), "| applicable:", sum(f["applicable"] for f in feats.values()))

file exists: True
videos: 400 | applicable: 397


In [11]:
import pandas as pd
FEATS = ["c0","cmax","cabs","cmean","lag_abs","valid_frac","speech_frac"]
rows = []
for part, ids in split.items():
    for v in ids:
        v = vid_id(v); f = feats.get(v)
        if f is None: continue
        r = dict(vid=v, part=part, y=label_of[v], applicable=f["applicable"])
        if f["applicable"]:
            r.update({k: f[k] for k in ["c0","cmax","cabs","cmean","valid_frac","speech_frac"]}); r["lag_abs"] = abs(f["lag_best"])
        rows.append(r)
df = pd.DataFrame(rows)
print(df.groupby(["part","y"]).applicable.agg(total="size", usable="sum"))
print(df[(df.part=="train") & df.applicable].groupby("y")[FEATS].mean().round(3))

         total  usable
part  y               
test  0     12      11
      1     42      41
train 0     53      53
      1    239     238
val   0     12      12
      1     42      42
      c0   cmax   cabs  cmean  lag_abs  valid_frac  speech_frac
y                                                              
0 -0.027  0.160  0.244  0.025    0.194       0.984        0.832
1 -0.032  0.156  0.252  0.012    0.194       0.980        0.764


In [12]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

SYNC = ["c0","cmax","cabs","cmean","lag_abs"]            # the real lip-sync features (final model)
ALL  = SYNC + ["valid_frac","speech_frac"]               # for comparison only

tr = df[(df.part=="train") & df.applicable]; va = df[(df.part=="val") & df.applicable]

print("Single-feature AUC on TRAIN (0.5 = no signal; 'fake' = higher):")
for k in ALL:
    print(f"  {k:12s} {roc_auc_score(tr.y, tr[k]):.3f}")

def fit_eval(cols):
    sc = StandardScaler().fit(tr[cols])
    m = LogisticRegressionCV(Cs=[0.01,0.1,1,10], cv=5, class_weight="balanced",
                             scoring="roc_auc", max_iter=2000).fit(sc.transform(tr[cols]), tr.y)
    p = m.predict_proba(sc.transform(va[cols]))[:,1]
    return sc, m, roc_auc_score(va.y, p), balanced_accuracy_score(va.y, (p>=0.5).astype(int))

for name, cols in [("SYNC only (final)", SYNC), ("ALL incl. artifacts (comparison)", ALL)]:
    sc, m, auc, bacc = fit_eval(cols)
    print(f"{name:34s} VAL AUC {auc:.3f} | bal acc {bacc:.3f}")
    if cols is SYNC: scaler, clf, FEATS = sc, m, SYNC   # keep the SYNC model for Steps 8–9

Single-feature AUC on TRAIN (0.5 = no signal; 'fake' = higher):
  c0           0.500
  cmax         0.496
  cabs         0.529
  cmean        0.488
  lag_abs      0.505
  valid_frac   0.495
  speech_frac  0.391
SYNC only (final)                  VAL AUC 0.669 | bal acc 0.673
ALL incl. artifacts (comparison)   VAL AUC 0.607 | bal acc 0.583


In [13]:
te = df[(df.part=="test") & df.applicable]
pt = clf.predict_proba(scaler.transform(te[FEATS]))[:,1]
y = te.y.values
auc = roc_auc_score(y, pt); bacc = balanced_accuracy_score(y, (pt>=0.5).astype(int))
rng = np.random.default_rng(0); boots = []
for _ in range(2000):
    i = rng.integers(0, len(y), len(y))
    if len(set(y[i])) == 2: boots.append(roc_auc_score(y[i], pt[i]))
lo, hi = np.percentile(boots, [2.5, 97.5])
print("TEST videos:", len(te), "| real:", int((y==0).sum()), "| fake:", int((y==1).sum()))
print(f"TEST AUC {auc:.3f}  (95% interval {lo:.2f}-{hi:.2f}) | balanced acc {bacc:.3f}")

TEST videos: 52 | real: 11 | fake: 41
TEST AUC 0.568  (95% interval 0.36-0.75) | balanced acc 0.496


In [16]:
import json, pickle
info = dict(features=FEATS,
            val_auc=float(roc_auc_score(va.y, clf.predict_proba(scaler.transform(va[FEATS]))[:,1])),
            test_auc=float(auc), test_auc_ci95=[float(lo), float(hi)],
            test_bal_acc=float(bacc), n_test=int(len(te)),
            note="No discriminative lip-sync signal found on DFDC (single-feature train AUC ~0.5). Fusion weight ~0.")

pickle.dump(dict(scaler=scaler, clf=clf, feats=FEATS), open("/content/drive/MyDrive/lipsync_model.pkl","wb"))
json.dump(info, open("/content/drive/MyDrive/lipsync_model_info.json","w"), indent=2)

def lipsync_score(video_id):
    f = process_video(video_id)
    if not f["applicable"]:
        return dict(score=None, confidence=0.0, applicable=False, reason=f["reason"])   # abstain, never "real"
    vals = {**f, "lag_abs": abs(f["lag_best"])}
    x = scaler.transform([[vals[k] for k in FEATS]])
    p = float(clf.predict_proba(x)[0,1])
    return dict(score=p, confidence=abs(p-0.5)*2, applicable=True, reason="")

print(info)
print(lipsync_score(list(label_of)[0]))

{'features': ['c0', 'cmax', 'cabs', 'cmean', 'lag_abs'], 'val_auc': 0.6686507936507936, 'test_auc': 0.5676274944567627, 'test_auc_ci95': [0.3585192065679871, 0.7500337837837836], 'test_bal_acc': 0.4955654101995566, 'n_test': 52, 'note': 'No discriminative lip-sync signal found on DFDC (single-feature train AUC ~0.5). Fusion weight ~0.'}
{'score': 0.48777362283997044, 'confidence': 0.02445275432005911, 'applicable': True, 'reason': ''}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
